# 06 - Config-Driven and Transform

> **When to use**: When you need to reuse configs, batch-generate multi-table data, or transform data after generation.
>
> **Core concept**: Pydantic config models + YAML/JSON files + Transform scripts + Snapshot.

## Applicable Scenarios

- Multi-table batch generation → YAML config + `fill_from_config()`
- Complex business logic (e.g., conditional computation) → Transform Script
- CI/CD reproducible test data → Snapshot + `replay()`
- Team-shared test environment → YAML config committed to Git

## What You Will Learn

- Config model hierarchy: GeneratorConfig → TableConfig → ColumnConfig
- YAML/JSON config formats
- Transform Scripts business logic
- ColumnAssociation cross-table association
- SnapshotManager snapshot management

**📚 Tutorial Navigation**

| No. | Topic | Architecture Layer | Prerequisites |
|------|------|--------|----------|
| 01 | Quick Start and Core Workflow | Orchestrator | None |
| 02 | 9-Level Strategy Chain | Core: ColumnMapper | 01 |
| 03 | Generators and Provider System | Generators | 01 |
| 04 | Database Layer and Multi-table | Database + Core | 01 |
| 05 | Expression Derivation and Constraint Solving | Core: DAG / Expression | 01 |
| **→ 06** | **Config-Driven and Transform** | **Config / Core** | **01** |
| 07 | AI Smart Config | Plugins: AI | 01 |
| 08 | MCP Server Integration | Plugins: MCP | 07 |
| 09 | Plugin System and Hook Lifecycle | Plugins | 01 |
| 10 | CLI Reference Manual | CLI | 06 |
| 11 | Utilities Reference | Utils | 01 |
| 12 | Testing Integration Patterns | Testing | 01 |

---

In [1]:
ORG_QUERY = "SELECT org_code FROM organizations"
ORG_PATTERN = "ORG-\\d{4}"
from sqlseed.config.models import GeneratorConfig, TableConfig, ColumnConfig, ProviderType, ColumnConstraintsConfig, ColumnAssociation
from sqlseed.config.loader import save_config, load_config, generate_template
from sqlseed.config.snapshot import SnapshotManager
from pathlib import Path

import sqlite3
# Prerequisite: pip install -e ".[dev,all]"
import sqlseed
from sqlseed import connect, fill, fill_from_config, preview

# Demo database setup
import sys
sys.path.insert(0, "..")  # for build_demo_db only
from build_demo_db import build
db_path = build()  # Force rebuild to ensure idempotent run

# Populate base dependencies
with connect(str(db_path)) as orch:
    orch.fill_table("organizations", count=5, seed=42)
    orch.fill_table("members", count=20, seed=42)
    orch.fill_table("projects", count=10, seed=42)
    orch.fill_table("tags", count=8, seed=42)

print(f"sqlseed {sqlseed.__version__} | Database: {db_path}")

Generating organizations:   0%|          | 0/5 [00:00<?, ?it/s]

Generating members:   0%|          | 0/20 [00:00<?, ?it/s]

Generating projects:   0%|          | 0/10 [00:00<?, ?it/s]

Generating tags:   0%|          | 0/8 [00:00<?, ?it/s]

sqlseed 0.1.16.dev1+g0824e8553.d20260505 | Database: /Users/sunbo/Documents/webblock/sqlseed/examples/sqlseed_demo.db


### 📍 Architecture Position

| Module | File | Core Class/Function |
|------|------|------------|
| Transform Loading | `src/sqlseed/core/transform.py` | `TransformLoader` |
| Config Model | `src/sqlseed/config/models.py` | `GeneratorConfig` |

> Corresponding architecture diagram: [§9 Config Model Hierarchy](../docs/architecture.zh-CN.md#9-配置模型层次结构)

## 1. See It in Action — YAML-Driven Batch Filling

For complex multi-table scenarios, declare all tables and columns in a YAML config file, then batch-fill with one line of code:

```yaml
db_path: "app.db"
tables:
  - name: users
    count: 10000
    columns:
      - name: email
        generator: email
```

In [2]:

# Build config with Python objects (equivalent to YAML)
config = GeneratorConfig(
    db_path=str(db_path),
    provider=ProviderType.MIMESIS,
    tables=[
        TableConfig(name="organizations", count=5, clear_before=True),
        TableConfig(name="members", count=10, clear_before=True),
    ]
)
config_path = Path("_demo_config.yaml")
save_config(config, str(config_path))

# One line to batch-fill multiple tables
results = fill_from_config(str(config_path))
print(f"{'Table':<15s}  {'Rows':>6s}  {'Time':>8s}  {'Speed':>10s}")
print('-' * 45)
for r in results:
    print(f"{r.table_name:<15s}  {r.count:>6d}  {r.elapsed:>7.3f}s  {r.rows_per_second:>8.0f} rows/s")

config_path.unlink(missing_ok=True)

Generating organizations:   0%|          | 0/5 [00:00<?, ?it/s]

Generating members:   0%|          | 0/10 [00:00<?, ?it/s]

Table                   Rows        Time          Speed
---------------------------------------------
organizations         5    0.031s       163 rows/s
members              10    0.033s       299 rows/s


Benefits of config-driven approach:

- **Version controllable** — YAML files can be committed to Git
- **Reproducible** — Same config = same data (with seed)
- **Shareable** — Team members use the same config

Below we break down each layer of the config model.

## 2. Config Model Hierarchy

sqlseed uses Pydantic models to define the config hierarchy:

```
GeneratorConfig
├── db_path: str
├── provider: ProviderType (MIMESIS)
├── locale: str (en_US)
├── tables: list[TableConfig]
│   └── TableConfig
│       ├── name: str
│       ├── count: int (1000)
│       ├── columns: list[ColumnConfig]
│       │   └── ColumnConfig
│       │       ├── Source mode: generator + params
│       │       └── Derived mode: derive_from + expression
│       └── clear_before, seed, transform, enrich
├── associations: list[ColumnAssociation]
└── optimize_pragma, log_level, snapshot_dir
```

In [3]:
config = GeneratorConfig(
    db_path=str(db_path),
    provider=ProviderType.MIMESIS,
    locale="en_US",
    tables=[
        TableConfig(
            name="organizations",
            count=5,
            columns=[
                ColumnConfig(name="name", generator="company"),
                ColumnConfig(name="description", generator="sentence"),
            ],
            clear_before=True,
        ),
    ],
)
print(f"GeneratorConfig: db_path={Path(config.db_path).name}")
print(f"  provider={config.provider}, locale={config.locale}")
print(f"  tables: {[t.name for t in config.tables]}")
print(f"  TableConfig[0]: count={config.tables[0].count}, columns={len(config.tables[0].columns)}")

GeneratorConfig: db_path=sqlseed_demo.db
  provider=ProviderType.MIMESIS, locale=en_US
  tables: ['organizations']
  TableConfig[0]: count=5, columns=2


## 3. ColumnConfig Dual-Mode Validation

ColumnConfig has two mutually exclusive modes:
- **Source mode**: `generator` + `params` + `null_ratio` + `provider`
- **Derived mode**: `derive_from` + `expression`

Pydantic `model_validator` enforces mutual exclusion; setting both `generator` and `derive_from` raises an error.

In [4]:
try:
    bad_config = ColumnConfig(
        name="test",
        generator="email",
        derive_from="other_col",
    )
except Exception as e:
    print(f"❌ Dual-mode conflict: {type(e).__name__}")
    print(f"   {e}")

print("\n✅ Source mode (generator):")
src = ColumnConfig(name="email_col", generator="email")
print(f"   generator={src.generator}, derive_from={src.derive_from}")

print("\n✅ Derived mode (derive_from):")
drv = ColumnConfig(name="short_code", derive_from="project_no", expression="value[-6:]")
print(f"   generator={drv.generator}, derive_from={drv.derive_from}, expression={drv.expression}")

❌ Dual-mode conflict: ValidationError
   1 validation error for ColumnConfig
  Value error, Column 'test': cannot use both 'generator' and 'derive_from' [type=value_error, input_value={'name': 'test', 'generat...rive_from': 'other_col'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error

✅ Source mode (generator):
   generator=email, derive_from=None

✅ Derived mode (derive_from):
   generator=None, derive_from=project_no, expression=value[-6:]


## 4. Complete YAML Config in Practice

In [5]:
config = GeneratorConfig(
    db_path=str(db_path),
    provider=ProviderType.MIMESIS,
    tables=[
        TableConfig(name="organizations", count=5, clear_before=True),
        TableConfig(
            name="members",
            count=20,
            clear_before=True,
            columns=[
                ColumnConfig(name="member_no", generator="pattern", params={"pattern": "M-\\d{6}"}, constraints=ColumnConstraintsConfig(unique=True)),  # noqa: E501
                ColumnConfig(name="email", generator="email", constraints=ColumnConstraintsConfig(unique=True)),
            ],
        ),
        TableConfig(
            name="projects",
            count=10,
            clear_before=True,
            columns=[
                ColumnConfig(name="project_no", generator="pattern", params={"pattern": "PRJ-\\d{6}"}, constraints=ColumnConstraintsConfig(unique=True)),  # noqa: E501
            ],
        ),
    ],
)
config_path = Path("_demo_config.yaml")
save_config(config, str(config_path))

results = fill_from_config(str(config_path))
for r in results:
    status = "✅" if r.count > 0 else "❌"
    print(f"{status} {r.table_name}: {r.count} rows in {r.elapsed:.3f}s")

Generating organizations:   0%|          | 0/5 [00:00<?, ?it/s]

Generating members:   0%|          | 0/20 [00:00<?, ?it/s]

Generating projects:   0%|          | 0/10 [00:00<?, ?it/s]

✅ organizations: 5 rows in 0.033s
✅ members: 20 rows in 0.059s
✅ projects: 10 rows in 0.043s


## 5. JSON Config Format

`save_config` and `load_config` support both YAML and JSON formats, auto-detected by file extension.

In [6]:
import json

json_path = Path("_demo_config.json")

save_config(config, str(json_path))
with open(json_path) as f:
    data = json.load(f)
print("JSON config format:")
safe = json.dumps(data, indent=2, ensure_ascii=False)
print(safe[:500] + "...")

json_path.unlink(missing_ok=True)

JSON config format:
{
  "db_path": "/Users/sunbo/Documents/webblock/sqlseed/examples/sqlseed_demo.db",
  "provider": "mimesis",
  "locale": "en_US",
  "tables": [
    {
      "name": "organizations",
      "count": 5,
      "batch_size": 5000,
      "columns": [],
      "clear_before": true,
      "seed": null,
      "transform": null,
      "enrich": false
    },
    {
      "name": "members",
      "count": 20,
      "batch_size": 5000,
      "columns": [
        {
          "name": "member_no",
          "genera...


## 6. load_config Demo

In [7]:
loaded = load_config(str(config_path))
print("load_config result:")
print(f"  db_path: {Path(loaded.db_path).name}")
print(f"  provider: {loaded.provider}")
print(f"  tables: {[t.name for t in loaded.tables]}")
print(f"  tables[1].columns: {[c.name for c in loaded.tables[1].columns]}")

load_config result:
  db_path: sqlseed_demo.db
  provider: ProviderType.MIMESIS
  tables: ['organizations', 'members', 'projects']
  tables[1].columns: ['member_no', 'email']


## 7. generate_template / sqlseed init

`generate_template()` auto-generates a config template based on DB schema; the CLI equivalent is `sqlseed init`.

In [8]:
template = generate_template(str(db_path), table_name="organizations")
print("generate_template result:")
print(f"  db_path: {Path(template.db_path).name}")
print(f"  tables: {[t.name for t in template.tables]}")
if template.tables:
    t = template.tables[0]
    print(f"  '{t.name}' columns ({len(t.columns)}):")
    for c in t.columns[:5]:
        print(f"    - {c.name}: generator={c.generator}")

generate_template result:
  db_path: sqlseed_demo.db
  tables: ['organizations']
  'organizations' columns (0):


## 8. Transform Scripts — Complex Business Logic

For business logic that declarative configs cannot express, use a Python Transform Script. The script must define a `transform_row(row, ctx)` function that transforms each row after generation and before writing to the database.

**Typical scenarios**:
- Conditional computation (e.g., compute VIP level by age)
- Data formatting (e.g., add international prefix to phone numbers)
- Field composition (e.g., concatenate full_name)
- Data validation and correction

In [9]:
transform_script = Path("_demo_transform.py")
transform_script.write_text(
    "def transform_row(row, ctx):\n"
    "    # Compute organization size tier by member_count\n"
    "    count = row.get('member_count', 0) or 0\n"
    "    if count >= 200:\n"
    '        row[\'description\'] = f"[Large] {row.get(\'name\', \'\')} - Global leading tech enterprise"\n'
    "    elif count >= 50:\n"
    '        row[\'description\'] = f"[Medium] {row.get(\'name\', \'\')} - Fast-growing tech company"\n'
    "    else:\n"
    '        row[\'description\'] = f"[Small] {row.get(\'name\', \'\')} - Innovative startup"\n'
    "    # Uppercase the name\n"
    "    if row.get('name'):\n"
    "        row['name'] = row['name'].upper()\n"
    "    return row\n"
)

with connect(str(db_path)) as orch:
    result = orch.fill_table("organizations", count=5, clear_before=True, transform=str(transform_script),
        columns={"org_code": {"type": "pattern", "regex": ORG_PATTERN},
                 "parent_code": {"type": "choice", "choices": [*[r[0] for r in __import__("sqlite3").connect(str(db_path)).execute(ORG_QUERY).fetchall()]]}})
    print(f"Transform fill: {result.count} rows")

conn = sqlite3.connect(str(db_path))
rows = conn.execute("SELECT name, member_count, description FROM organizations").fetchall()
for r in rows:
    desc = str(r[2])[:60]
    print(f"  {r[0]:<20s} | members={r[1]:4d} | {desc}")
conn.close()

transform_script.unlink(missing_ok=True)


Generating organizations:   0%|          | 0/5 [00:00<?, ?it/s]

Transform fill: 5 rows
  DOUGLASS HOUSE       | members=   0 | [Small] Douglass House - Innovative startup
  SONG OLSEN           | members=   0 | [Small] Song Olsen - Innovative startup
  ANTONINA TERRY       | members=   0 | [Small] Antonina Terry - Innovative startup
  VI HINES             | members=   0 | [Small] Vi Hines - Innovative startup
  LYNDIA SAUNDERS      | members=   0 | [Small] Lyndia Saunders - Innovative startup


## 9. ColumnAssociation Cross-Table Association

`ColumnAssociation` declares that multiple tables share the same column values, ensuring FK reference consistency.

In [10]:
config_with_assoc = GeneratorConfig(
    db_path=str(db_path),
    tables=[
        TableConfig(name="organizations", count=3, clear_before=True, columns=[
            ColumnConfig(name="org_code", generator="pattern", params={"regex": ORG_PATTERN}),
            ColumnConfig(name="parent_code", generator="choice", params={"choices": [*[r[0] for r in __import__("sqlite3").connect(str(db_path)).execute(ORG_QUERY).fetchall()]]}),
        ]),
        TableConfig(name="members", count=10, clear_before=True),
    ],
    associations=[
        ColumnAssociation(
            column_name="org_code",
            source_table="organizations",
            target_tables=["members"],
            strategy="shared_pool",
        ),
    ],
)
assoc_path = Path("_assoc_config.yaml")
save_config(config_with_assoc, str(assoc_path))

results = fill_from_config(str(assoc_path))
for r in results:
    print(f"  {r.table_name}: {r.count} rows")

conn = sqlite3.connect(str(db_path))
org_codes = [r[0] for r in conn.execute("SELECT DISTINCT org_code FROM organizations").fetchall()]
member_orgs = [r[0] for r in conn.execute("SELECT org_code FROM members LIMIT 5").fetchall()]
print(f"\nAssociation validation: org_codes={org_codes}")
print(f"  members org_code: {member_orgs}")
all_valid = all(m in org_codes for m in member_orgs if m)
print(f"  All FK references valid: {all_valid}")
conn.close()

assoc_path.unlink(missing_ok=True)


Generating organizations:   0%|          | 0/3 [00:00<?, ?it/s]

Generating members:   0%|          | 0/10 [00:00<?, ?it/s]

  organizations: 3 rows
  members: 10 rows

Association validation: org_codes=['ORG-2543', 'ORG-2948', 'ORG-8473']
  members org_code: ['ORG-2948', 'ORG-8473', 'ORG-8473', 'ORG-8473', 'ORG-8473']
  All FK references valid: True


## 10. ColumnConstraintsConfig Constraint Configuration

`ColumnConstraintsConfig` supports the following constraints:
- `unique`: unique constraint
- `min_value` / `max_value`: numeric range
- `regex`: regex matching
- `max_retries`: max retry count

In [11]:
constrained = GeneratorConfig(
    db_path=str(db_path),
    tables=[
        TableConfig(
            name="organizations",
            count=5,
            clear_before=True,
            columns=[
                ColumnConfig(name="name", generator="company", constraints=ColumnConstraintsConfig(unique=True)),
                ColumnConfig(name="org_code", generator="pattern", params={"pattern": ORG_PATTERN}, constraints=ColumnConstraintsConfig(unique=True)),  # noqa: E501
            ],
        ),
    ],
)
c_path = Path("_constraints_config.yaml")
save_config(constrained, str(c_path))

results = fill_from_config(str(c_path))
for r in results:
    print(f"  {r.table_name}: {r.count} rows")

conn = sqlite3.connect(str(db_path))
names = [r[0] for r in conn.execute("SELECT name FROM organizations").fetchall()]
codes = [r[0] for r in conn.execute(ORG_QUERY).fetchall()]
print(f"\nUNIQUE name: {len(names) == len(set(names))}")
print(f"UNIQUE org_code: {len(codes) == len(set(codes))}")
conn.close()

c_path.unlink(missing_ok=True)
config_path.unlink(missing_ok=True)

Generating organizations:   0%|          | 0/5 [00:00<?, ?it/s]

  organizations: 5 rows

UNIQUE name: True
UNIQUE org_code: True


## 11. SnapshotManager Snapshot Management

SnapshotManager supports saving, loading, listing, and replaying data snapshots; the CLI equivalent is `sqlseed replay`.

In [12]:

snap_mgr = SnapshotManager()  # Uses platform cache dir (~/Library/Caches/sqlseed/snapshots on macOS)

config = GeneratorConfig(
    db_path=str(db_path),
    tables=[TableConfig(name="organizations", count=3, clear_before=True)],
)
snapshot_path = snap_mgr.save(config, "organizations", count=3, seed=42)
print(f"Snapshot saved: {Path(snapshot_path).name}")
print(f"Snapshot dir: {snap_mgr._snapshot_dir}")

snapshots = snap_mgr.list_snapshots()
print(f"Existing snapshots: {len(snapshots)}")

data = snap_mgr.load(snapshot_path)
print(f"Snapshot content: table={data.get('table_name')}, count={data.get('count')}")

# Replay snapshot — fully reproduces the previous generation
result = snap_mgr.replay(snapshot_path)
print(f"\nSnapshot replay: {result}")

Snapshot saved: 2026-05-06_074253_organizations.yaml
Snapshot dir: /Users/sunbo/Library/Caches/sqlseed/snapshots
Existing snapshots: 77
Snapshot content: table=organizations, count=3


Generating organizations:   0%|          | 0/3 [00:00<?, ?it/s]


Snapshot replay: GenerationResult(table=organizations, count=3, elapsed=0.04s, speed=85.51 rows/s)


### CLI Snapshot Commands

```bash
# Generate and save snapshot
sqlseed fill app.db --table users --count 10000 --seed 42 --snapshot
# → Snapshot saved: snapshots/2026-04-15_033000_users.yaml

# Replay snapshot
sqlseed replay snapshots/2026-04-15_033000_users.yaml
```

**Typical scenarios**: CI/CD reproducible test data, team-shared consistent test environment, fast database state rebuild.

## 12. Preview & Debug CLI

`sqlseed preview` previews data without writing; `sqlseed inspect --show-mapping` shows column mapping strategies.

In [13]:
from click.testing import CliRunner

from sqlseed.cli.main import cli

preview_rows = preview(str(db_path), table="organizations", count=3,
                       columns={"org_code": {"type": "pattern", "regex": ORG_PATTERN},
                                "name": {"type": "company"}})
print("preview result:")
for row in preview_rows:
    print(f"  {row.get('org_code', 'N/A')} | {row.get('name', 'N/A')}")



runner = CliRunner()
result = runner.invoke(cli, ["inspect", str(db_path), "-t", "organizations", "--show-mapping"])
if result.output.strip():
    print("\nsqlseed inspect --show-mapping:")
    print(result.output[:500])

                                           Table: organizations (3 rows)                                           
┏━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━┳━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Column       ┃ Type        ┃ Nullable ┃ PK ┃ Auto ┃ Generator   ┃ Params                                        ┃
┡━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━╇━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ org_code     │ VARCHAR(16) │ ✗        │ ✓  │      │ string      │ {'min_length': 6, 'max_length': 12,           │
│              │             │          │    │      │             │ 'charset': 'alphanumeric'}                    │
│ name         │ VARCHAR(64) │ ✗        │    │      │ name        │ {}                                            │
│ parent_code  │ VARCHAR(16) │ ✓        │    │      │ foreign_key │ {'ref_table': 'organizations', 'ref_column':  │
│              │             │          │    │      │             │ 'org_code', 'strategy': 'random',             │
│              │             │          │    │      │             │ '_ref_values': ['JmTPSI', 'fLBcbfnoGM',       │
│              │             │          │    │      │             │ 'hbVrpoiVgRV']}                               │
│ description  │ TEXT        │ ✓        │    │      │ text        │ {'min_length': 100, 'max_length': 500}        │
│ is_active    │ INTEGER     │ ✓        │    │      │ skip        │ {}                                            │
│ member_count │ INTEGER     │ ✓        │    │      │ skip        │ {}                                            │
│ created_at   │ TEXT        │ ✓        │    │      │ datetime    │ {}                                            │
└──────────────┴─────────────┴──────────┴────┴──────┴─────────────┴───────────────────────────────────────────────┘

        Foreign Keys: organizations         
┏━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┓
┃ Column      ┃ Ref Table     ┃ Ref Column ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━┩
│ parent_code │ organizations │ org_code   │
└─────────────┴───────────────┴────────────┘

preview result:
  ORG-9977 | Enterasys Networks
  ORG-6810 | American Broadcasting Company
  ORG-5959 | Alienware


## ✅ Summary

| Feature | API | Status |
|---|---|---|
| Config Model Hierarchy | GeneratorConfig/TableConfig/ColumnConfig | ✅ |
| Dual-Mode Validation | ColumnConfig model_validator | ✅ |
| YAML/JSON Config | save_config/load_config | ✅ |
| Auto Template Generation | generate_template / sqlseed init | ✅ |
| Transform Scripts | transform_row(row, ctx) | ✅ |
| Cross-Table Association | ColumnAssociation | ✅ |
| Constraint Config | ColumnConstraintsConfig | ✅ |
| Snapshot Management | SnapshotManager | ✅ |
| Preview & Debug | preview / inspect --show-mapping | ✅ |

**Next**: [07-ai-plugin.ipynb](07-ai-plugin.ipynb) — AI Smart Config

In [14]:
# ✅ Validation: ensure data was successfully generated and written
import sqlite3
conn = sqlite3.connect(str(db_path))
try:
    # Basic row count validation
    member_count = conn.execute("SELECT COUNT(*) FROM members").fetchone()[0]
    assert member_count > 0, f"Expected members > 0, got {member_count}"
    print("✅ All assertions passed")
finally:
    conn.close()

✅ All assertions passed
